## plot_emg: plotting miRPNI data in Python

Python/matplotlib equivalent of `plot_emg.m`. Builds `trial_meta` from the CSV + metadata JSON pipeline (see `csv_to_dataframe.ipynb` for a full walkthrough of that step), then plots either:

- a **single trial** — useful for checking raw signal quality, checking for noise or artifacts in one recording, or
- the **mean across all trials of a task** — useful for seeing the underlying gesture-evoked shape once trial-to-trial noise has been averaged out.

**How to use this notebook:** run every cell top to bottom once to see both modes demonstrated on the sample session. After that, edit the values in **Settings** to point at your own session/trial/task and re-run.

**Requires** `mirpni_utils.py` in the same folder as this notebook (see the repo README).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mirpni_utils import build_trial_meta_from_csv, ordered_channel_names

Matplotlib is building the font cache; this may take a moment.


### Settings

#### Which session to load
These four paths point at one session's worth of files: the raw EMG CSV, that session's trial metadata, the participant's channel-name lookup, and the dataset-wide task-name lookup. Change these to look at a different session; the defaults load the sample session shipped in this repo.

In [ ]:
DATA_PATH = "sample_set/csv/P1_S12_EMG1kHz.csv"
TRIAL_META_PATH = "sample_set/meta/P1_S12_meta.json"
CH_META_PATH = "sample_set/meta/P1_metadata.json"
TASKS_PATH = "sample_set/movements.json"

FS = 1000            # sampling rate (Hz)

#### What to plot
Once the session above is loaded, these three values control what you'll actually see. `TRIAL_ID` and `TASK_NUMBER` are independent: only one of these variables is used depending on `PLOT_MEAN`:

| `PLOT_MEAN` | Which setting is used | What you get |
|---|---|---|
| `False` | `TRIAL_ID` | one specific trial, as recorded |
| `True` | `TASK_NUMBER` | the average of every trial belonging to that task |

In [ ]:
TRIAL_ID = 53         # which trial to plot (ignored when PLOT_MEAN = True)
TASK_NUMBER = 1        # which task to average over (used when PLOT_MEAN = True)
PLOT_MEAN = False      # False = single trial | True = mean across task trials

## Load metadata and assemble `trial_meta`

`build_trial_meta_from_csv` and `ordered_channel_names` do the same reshape and channel-lookup work walked through step by step in `csv_to_dataframe.ipynb` — see that notebook if you want to understand *how* this works. They're imported here rather than redefined, so this notebook and `csv_to_dataframe.ipynb` can never quietly disagree about what that reshape produces.

Running the cell below should print a channel list, a trial/task count, and the first few rows of `trial_meta` — if that looks right, the paths above are correct and you're ready to plot.

In [ ]:
trial_meta, n_ch = build_trial_meta_from_csv(DATA_PATH, TRIAL_META_PATH, TASKS_PATH)
channel_names = ordered_channel_names(CH_META_PATH, n_ch)

print(f"Channels : {', '.join(channel_names)}")
print(f"Trials   : {len(trial_meta)} | Tasks: {sorted(trial_meta['TaskNumber'].unique())}")
trial_meta.head()

## Plotting functions

Two functions, one per mode from the settings table above. Both take the same `trial_meta` and `channel_names` and return a stacked grid of subplots, one row per EMG channel, so you can compare the timing and amplitude of muscle activity across channels directly.

#### Single trial
Plots the raw, unaveraged signal for one trial — what a single recording actually looked like including the noise.

In [ ]:
def plot_single_trial(trial_meta, trial_id, channel_names, fs=FS):
    """Plot every channel for a single trial as a stacked grid of subplots."""
    row = trial_meta.loc[trial_meta['TrialID'] == trial_id].iloc[0]
    emg = row['EMG1k']  # (numSamples, numChannels)
    n_samples, n_ch = emg.shape
    time = np.arange(n_samples) / fs
    title = (
        f"EMG — Trial {trial_id} "
        f"(Task {row['TaskNumber']}: {row['TaskName']}, Rep {row['TrialNumber']})"
    )

    fig, axes = plt.subplots(n_ch, 1, figsize=(9, 11), sharex=True)
    for ci, ax in enumerate(axes):
        ax.plot(time, emg[:, ci], linewidth=0.6)
        ax.set_xlim(0, time.max())
        ax.grid(True, color='0.85')
        ax.set_ylabel(channel_names[ci], fontsize=8, rotation=0, ha='right', va='center')
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    fig.tight_layout()
    return fig

Preview it on `TRIAL_ID` from the settings above:

In [ ]:
fig = plot_single_trial(trial_meta, TRIAL_ID, channel_names)
plt.show()

### Mean across trials

Averages every trial for a given task together. Trial-to-trial noise tends to cancel out, leaving a cleaner view of the gesture-evoked activity pattern — useful for comparing how different muscles respond to a given task.

In [ ]:
def plot_mean_task(trial_meta, task_number, channel_names, fs=FS):
    """Plot the mean EMG trace (averaged across trials) for a given task."""
    subset = trial_meta.loc[trial_meta['TaskNumber'] == task_number]
    n_trials = len(subset)
    arrays = np.stack(subset['EMG1k'].to_list())  # (n_trials, numSamples, numChannels)
    emg_mean = arrays.mean(axis=0)
    n_samples, n_ch = emg_mean.shape
    time = np.arange(n_samples) / fs
    task_name = subset['TaskName'].iloc[0]
    title = f"Mean EMG — Task {task_number}: {task_name} (n={n_trials} trials)"

    fig, axes = plt.subplots(n_ch, 1, figsize=(9, 11), sharex=True)
    for ci, ax in enumerate(axes):
        ax.plot(time, emg_mean[:, ci], linewidth=0.6)
        ax.set_xlim(0, time.max())
        ax.grid(True, color='0.85')
        ax.set_ylabel(channel_names[ci], fontsize=8, rotation=0, ha='right', va='center')
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    fig.tight_layout()
    return fig

Preview it on `TASK_NUMBER` from the settings above:

In [ ]:
fig = plot_mean_task(trial_meta, TASK_NUMBER, channel_names)
plt.show()

## Putting it together

For everyday use you don't need both previews above — flip `PLOT_MEAN` in **Settings** and re-run just this cell to get whichever mode you want.

In [ ]:
if PLOT_MEAN:
    fig = plot_mean_task(trial_meta, TASK_NUMBER, channel_names)
else:
    fig = plot_single_trial(trial_meta, TRIAL_ID, channel_names)
plt.show()